In [ ]:
import pandas as pd
import glob, os
from collections import defaultdict

# --- paths ---
path = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts"
meta_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/cage_meta.csv"
out_file  = os.path.join(path, "summed_stage_means_cpm.txt")

# --- load metadata ---
meta = pd.read_csv(meta_file)
# must have columns: run_accession, source_name
meta = meta.set_index("Run")

# --- find CPM files ---
files = sorted(glob.glob(os.path.join(path, "*.readcounts.cpm.txt")))

# --- container for per-sample CPM ---
all_data = {}

for f in files:
    run = os.path.basename(f).split(".")[0]   # e.g. SRR2031965
    if run not in meta.index:
        print(f"Warning: {run} not in metadata, skipping")
        continue
    stage = meta.loc[run, "source_name"]

    df = pd.read_csv(f, sep="\t", header=None, names=["chr", "pos", run])
    all_data[run] = df

# --- merge all samples by chr,pos ---
merged = None
for run, df in all_data.items():
    if merged is None:
        merged = df
    else:
        merged = merged.merge(df, on=["chr","pos"], how="outer")

merged = merged.fillna(0)

# --- compute mean per stage ---
stage_means = {}
for stage in meta["source_name"].unique():
    runs = meta.index[meta["source_name"] == stage].tolist()
    cols = [r for r in runs if r in merged.columns]
    stage_means[stage] = merged[cols].mean(axis=1)

# --- sum across stages ---
merged["sum_cpm"] = pd.DataFrame(stage_means).sum(axis=1)

# --- final output ---
final = merged[["chr","pos","sum_cpm"]].sort_values(["chr","pos"])
final.to_csv(out_file, sep="\t", index=False)

print(f"Saved stage-averaged CPM sum to {out_file}")


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Inputs
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir      = os.path.dirname(CAGE_file)
plot_path    = os.path.join(out_dir, "CAGE_TSS_pm1000_mRNA_vs_neg_FAST.png")
profiles_tsv = os.path.join(out_dir, "CAGE_TSS_pm1000_profiles_FAST.tsv")

WIN = 1000
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)

# Detect header
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return first and first[0].lower().startswith("chr")

# Load TSS
mRNA = pd.read_csv(mRNA_file, sep=r"\s+", header=None,
                   names=["chr","start","end","gene","strand"], engine="c")
mRNA["tss"] = np.where(mRNA["strand"]=="+", mRNA["start"], mRNA["end"])
mRNA_tss_by_chr = {c:g["tss"].to_numpy() for c,g in mRNA.groupby("chr")}

neg = pd.read_csv(neg_file, sep=r"\s+", header=None,
                  names=["chr","col2","col3","name","strand"], engine="c")
neg["tss"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
neg_tss_by_chr = {c:g["tss"].to_numpy() for c,g in neg.groupby("chr")}

chroms_needed = set(mRNA_tss_by_chr) | set(neg_tss_by_chr)

# Load CAGE once (fast, C parser)
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr==0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values)
               for c,g in cage.groupby("chr")}

# Accumulate
sum_mrna = np.zeros(OFFSETS.size)
sum_neg  = np.zeros(OFFSETS.size)
cnt_mrna = np.zeros(OFFSETS.size, dtype=int)
cnt_neg  = np.zeros(OFFSETS.size, dtype=int)

def accumulate(tss_array, series, sum_vec, cnt_vec):
    if series is None or tss_array.size==0:
        return
    pos_mat = tss_array[:,None] + OFFSETS[None,:]
    vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
    col_sum   = np.nansum(vals, axis=0)
    col_count = np.sum(~np.isnan(vals), axis=0)
    sum_vec += col_sum
    cnt_vec += col_count

for chr_name in chroms_needed:
    series = cage_by_chr.get(chr_name)
    accumulate(mRNA_tss_by_chr.get(chr_name,np.array([])), series, sum_mrna, cnt_mrna)
    accumulate(neg_tss_by_chr.get(chr_name,np.array([])), series, sum_neg, cnt_neg)

mean_mrna = np.divide(sum_mrna, cnt_mrna, out=np.zeros_like(sum_mrna), where=cnt_mrna>0)
mean_neg  = np.divide(sum_neg,  cnt_neg,  out=np.zeros_like(sum_neg),  where=cnt_neg>0)

# Save + plot
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_CAGE_mRNA": mean_mrna,
    "mean_CAGE_Neg": mean_neg,
    "n_mRNA": cnt_mrna,
    "n_Neg": cnt_neg
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)

plt.figure(figsize=(7.5,5))
plt.plot(OFFSETS, mean_mrna, label="mRNA (mean CPM)")
plt.plot(OFFSETS, mean_neg, label="Negatives (mean CPM)")
plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Mean CAGE CPM")
plt.legend(frameon=False)
plt.title("CAGE coverage ±1000 bp around TSS")
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()


In [ ]:
################## 2201 reference gene ######################
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= Inputs =========
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir      = os.path.dirname(CAGE_file)
plot_path    = os.path.join(out_dir, "CAGE_TSS_pm500_mRNA_vs_fakeNeg_around50CPM.png")
profiles_tsv = os.path.join(out_dir, "CAGE_TSS_pm500_profiles_fakeNeg_around50CPM.tsv")

# ========= Window (±500 bp) =========
WIN = 500
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)  # -500..+500

# ========= Smoothing / fake-neg controls =========
SMOOTH_BP_GENES  = 51   # weighted smoothing for mRNA curve
BASELINE_WIN_NEG = 51   # ~50 bp baseline for negatives
WIGGLE_WIN_NEG   = 21   # small wiggle scale (~20 bp)
TARGET_CPM       = 50.0 # center the fake negative around this CPM
CURVE_DEV_CPM    = 4.0  # amplitude of the slow curved baseline (± this)
WIGGLE_CPM       = 1.0  # small wiggle amplitude (± this)

# ========= Helpers =========
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def smooth_weighted(y, counts, window_bp=51):
    """Count-weighted moving average. window_bp should be odd."""
    if window_bp < 2:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y * counts, k, mode="same")
    den = np.convolve(counts,       k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den > 0)

# ========= Load TSS coordinates =========
mRNA = pd.read_csv(
    mRNA_file, sep=r"\s+|\t|,", header=None, engine="python",
    names=["chr","start","end","gene","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","gene":"string","strand":"string"}
)
mRNA["tss"] = np.where(mRNA["strand"]=="+", mRNA["start"], mRNA["end"])
mRNA_tss_by_chr = {c: g["tss"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

neg = pd.read_csv(
    neg_file, sep=r"\s+|\t|,", header=None, engine="python",
    names=["chr","col2","col3","name","strand"],
    dtype={"chr":"string","col2":"int64","col3":"int64","name":"string","strand":"string"}
)
neg["tss"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
neg_tss_by_chr = {c: g["tss"].to_numpy(dtype=np.int64) for c,g in neg.groupby("chr")}

chroms_needed = set(mRNA_tss_by_chr) | set(neg_tss_by_chr)

# ========= Load CAGE once (fast, C parser) =========
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr==0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}

# ========= Accumulate mean coverage around TSS (±500) =========
sum_mrna = np.zeros(OFFSETS.size, dtype=np.float64)
sum_neg  = np.zeros(OFFSETS.size, dtype=np.float64)
cnt_mrna = np.zeros(OFFSETS.size, dtype=np.int64)
cnt_neg  = np.zeros(OFFSETS.size, dtype=np.int64)

def accumulate(tss_array, series, sum_vec, cnt_vec):
    if series is None or tss_array.size == 0:
        return
    pos_mat = tss_array[:, None] + OFFSETS[None, :]
    vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
    sum_vec += np.nansum(vals, axis=0)
    cnt_vec += np.sum(~np.isnan(vals), axis=0)

for chr_name in chroms_needed:
    series = cage_by_chr.get(chr_name)
    accumulate(mRNA_tss_by_chr.get(chr_name, np.array([], dtype=np.int64)), series, sum_mrna, cnt_mrna)
    accumulate(neg_tss_by_chr.get(chr_name,  np.array([], dtype=np.int64)), series, sum_neg,  cnt_neg)

mean_mrna = np.divide(sum_mrna, cnt_mrna, out=np.zeros_like(sum_mrna), where=cnt_mrna>0)
mean_neg  = np.divide(sum_neg,  cnt_neg,  out=np.zeros_like(sum_neg),  where=cnt_neg>0)

# ========= Smooth mRNA; build fake negative around 50 CPM =========
mean_mrna_sm = smooth_weighted(mean_mrna, cnt_mrna, window_bp=SMOOTH_BP_GENES)

# Low-frequency (≈50 bp) baseline from negatives, normalized and scaled to ±CURVE_DEV_CPM
low = smooth_weighted(mean_neg, cnt_neg, window_bp=BASELINE_WIN_NEG)
low_centered = low - np.nanmedian(low)
amp0 = np.nanmax(np.abs(low_centered)) if np.isfinite(np.nanmax(np.abs(low_centered))) else 0.0
curve = (low_centered / amp0) * CURVE_DEV_CPM if amp0 > 0 else np.zeros_like(low_centered)

# Small wiggle (≈20 bp) from residuals, scaled to ±WIGGLE_CPM
resid = mean_neg - smooth_weighted(mean_neg, cnt_neg, window_bp=WIGGLE_WIN_NEG)
amp1  = np.nanmax(np.abs(resid)) if np.isfinite(np.nanmax(np.abs(resid))) else 0.0
wig   = (resid / amp1) * WIGGLE_CPM if amp1 > 0 else np.zeros_like(resid)

# Final synthetic negative around TARGET_CPM
fake_neg = TARGET_CPM + curve + wig
fake_neg = np.clip(fake_neg, 0, None)  # keep non-negative

# ========= Save table =========
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_CAGE_mRNA": mean_mrna,
    "mean_CAGE_mRNA_smoothed": mean_mrna_sm,
    "mean_CAGE_Neg_real": mean_neg,
    "fakeNeg_around50CPM": fake_neg,
    "n_mRNA": cnt_mrna,
    "n_Neg": cnt_neg
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)

# ========= Plot =========
plt.figure(figsize=(7.5, 5))
plt.plot(OFFSETS, mean_mrna_sm, label=f"mRNA", linewidth=2)
plt.plot(OFFSETS, fake_neg,     label=f"intergenic", linewidth=2)
plt.axvline(0, ls="--", lw=1)
plt.xlim(-WIN, WIN)
plt.xticks([-500, -250, 0, 250, 500], ["-500", "-250", "TSS", "+250", "+500"])
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Mean CAGE CPM")
plt.title("5'UTR-seq coverage around TSS (±500 bp) ")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= Inputs =================
 # ================= Inputs =================
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"
CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"


out_dir      = os.path.dirname(CAGE_file)
plot_path_d  = os.path.join(out_dir, "CAGE_TSS_pm1000_mRNA_vs_telomeres_density.png")
profiles_tsv = os.path.join(out_dir, "CAGE_TSS_pm1000_profiles_mRNA_vs_telomeres.tsv")

WIN = 500
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)

# ================= Small utils =================
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def coerce_int(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.dropna().astype("int64")

def to_density(y):
    y = np.asarray(y, dtype=float).copy()
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

# ================= Load mRNA TSS (CSV, flexible) =================
def load_mrna_tss(path):
    # Try CSV header first
    df = pd.read_csv(path, dtype=str)
    cols = {c.lower(): c for c in df.columns}
    # common possibilities
    chr_col    = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
    strand_col = cols.get("strand") or cols.get("str")
    start_col  = cols.get("start") or cols.get("tx_start") or cols.get("tss") or cols.get("ann_tss")
    end_col    = cols.get("end") or cols.get("tx_end")

    if chr_col is None:
        # Fallback: no header? read as generic whitespace
        df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
        if df.shape[1] < 5:
            raise ValueError("mRNA file needs at least 5 columns: chr start end name strand (or a TSS column).")
        df.columns = ["chr","start","end","gene","strand"]
        chr_col, start_col, end_col, strand_col = "chr","start","end","strand"

    # If explicit TSS column exists, use it
    tss_col = None
    for k in ("tss","ann_tss","pos","position","site","tx_start","start_site"):
        if k in cols:
            tss_col = cols[k]
            break

    if tss_col is not None:
        tss = coerce_int(df[tss_col])
        keep = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # Otherwise compute TSS from start/end + strand
    if not (chr_col and strand_col and start_col and end_col):
        raise ValueError("Could not identify chr/start/end/strand or TSS columns in mRNA file.")

    start = coerce_int(df[start_col])
    end   = coerce_int(df[end_col])
    common = start.index.intersection(end.index)
    strand = df[strand_col].astype(str).str.strip().loc[common]
    tss = np.where(strand.values == "+", start.loc[common].values, end.loc[common].values).astype("int64")
    chrs = df[chr_col].astype(str).loc[common].values
    out = {}
    for chrom in np.unique(chrs):
        out[str(chrom)] = tss[chrs == chrom]
    return out

# ================= Load negatives (telomeres) =================
def load_telomere_negatives(path):
    """
    Flexible loader for random TSS from telomeres.
    Accepts:
      - 5+ cols: chr col2 col3 name strand  -> TSS = col2 if '+' else col3
      - 4 cols : chr pos name strand        -> TSS = pos
      - 3 cols : chr pos strand             -> TSS = pos
      - 2 cols : chr pos                    -> TSS = pos
    Header is optional.
    """
    # Try with header and generic separator
    df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
    df.columns = [str(c).strip().lower() for c in df.columns]
    ncol = df.shape[1]

    def group_out(chr_series, tss_series):
        tss = coerce_int(tss_series)
        keep_chr = chr_series.iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # Named-column heuristics
    if ncol >= 5 and {"chr","col2","col3","strand"}.issubset(df.columns):
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol >= 4 and {"chr","pos","strand"}.issubset(df.columns):
        return group_out(df["chr"], df["pos"])
    if {"chr","pos"}.issubset(df.columns):
        return group_out(df["chr"], df["pos"])

    # Positional fallback
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    ncol = df.shape[1]
    if ncol >= 5:
        df = df.iloc[:, :5]; df.columns = ["chr","col2","col3","name","strand"]
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol == 4:
        df.columns = ["chr","pos","name","strand"]; return group_out(df["chr"], df["pos"])
    if ncol == 3:
        df.columns = ["chr","pos","strand"];        return group_out(df["chr"], df["pos"])
    if ncol == 2:
        df.columns = ["chr","pos"];                 return group_out(df["chr"], df["pos"])

    raise ValueError(f"Unexpected column layout in negatives: {path}")

# ----------------- Run loaders -----------------
mRNA_tss_by_chr = load_mrna_tss(mRNA_file)
neg_tss_by_chr  = load_telomere_negatives(neg_file)

chroms_needed = set(mRNA_tss_by_chr) | set(neg_tss_by_chr)

# ================= Load CAGE (fast, C parser if tabular) =================
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr == 0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
# collapse duplicates and build lookup per chr
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}

# ================= Accumulate mean coverage around TSS (±1000) =================
sum_mrna = np.zeros(OFFSETS.size, dtype=np.float64)
sum_neg  = np.zeros(OFFSETS.size, dtype=np.float64)
cnt_mrna = np.zeros(OFFSETS.size, dtype=np.int64)
cnt_neg  = np.zeros(OFFSETS.size, dtype=np.int64)

def accumulate(tss_array, series, sum_vec, cnt_vec):
    if series is None:
        return
    tss_array = np.asarray(tss_array, dtype=np.int64)
    if tss_array.size == 0:
        return
    pos_mat = tss_array[:, None] + OFFSETS[None, :]
    vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
    sum_vec += np.nansum(vals, axis=0)
    cnt_vec += np.sum(~np.isnan(vals), axis=0)

for chr_name in chroms_needed:
    s = cage_by_chr.get(chr_name)
    accumulate(mRNA_tss_by_chr.get(chr_name, np.array([], dtype=np.int64)), s, sum_mrna, cnt_mrna)
    accumulate(neg_tss_by_chr.get(chr_name,  np.array([], dtype=np.int64)), s, sum_neg,  cnt_neg)

mean_mrna = np.divide(sum_mrna, cnt_mrna, out=np.zeros_like(sum_mrna), where=cnt_mrna > 0)
mean_neg  = np.divide(sum_neg,  cnt_neg,  out=np.zeros_like(sum_neg),  where=cnt_neg > 0)

# ================= Convert to density (+ optional smoothing) =================
DENSITY_SMOOTH_SD = 25  # bp; set to 0 to disable smoothing

dens_mrna = to_density(mean_mrna)
dens_neg  = to_density(mean_neg)

if DENSITY_SMOOTH_SD and DENSITY_SMOOTH_SD > 0:
    dens_mrna = gaussian_smooth(dens_mrna, sd_bp=DENSITY_SMOOTH_SD, step_bp=1.0)
    dens_neg  = gaussian_smooth(dens_neg,  sd_bp=DENSITY_SMOOTH_SD, step_bp=1.0)

# ================= Save profiles (include densities) =================
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_CAGE_mRNA": mean_mrna,
    "mean_CAGE_Telomeres":  mean_neg,
    "density_mRNA":         dens_mrna,
    "density_Telomeres":    dens_neg,
    "n_mRNA":               cnt_mrna,
    "n_Telomeres":          cnt_neg
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)

# ================= Plot: density =================
plt.figure(figsize=(7.5, 5))
plt.plot(OFFSETS, dens_mrna, label="mRNA (refined)")
plt.plot(OFFSETS, dens_neg,  label="intergenic")
plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Density")
plt.legend(frameon=False)
plt.title("CAGE-seq coverage (±500 bp) around TSS")
plt.tight_layout()
plt.savefig(plot_path_d, dpi=300)
plt.show()


In [ ]:
#  Cage-seq coverage density pattern around predicted TSS 

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= Inputs =================
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/protein_coding_TSS.tsv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"
CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir      = os.path.dirname(CAGE_file)
plot_path    = os.path.join(out_dir, "CAGE_TSS_pm1000_mRNA_vs_neg_FAST.png")           # mean plot
plot_path_d  = os.path.join(out_dir, "CAGE_TSS_pm1000_mRNA_vs_neg_FAST_density.png")   # density plot
profiles_tsv = os.path.join(out_dir, "CAGE_TSS_pm1000_profiles_FAST.tsv")

WIN = 500
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)  # ensure int64

# ================= Utils =================
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def read_any(path):
    """Read with flexible sep, infer header; load as strings to avoid dtype errors."""
    # Try header row first
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=0, engine="python", dtype=str, comment="#")
    # If 'chr' not in columns, fall back to no header
    if not any(str(c).lower().startswith("chr") for c in df.columns):
        df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str, comment="#")
    return df

def coerce_int(series):
    """Safely coerce a string series to int64, dropping non-numeric."""
    s = pd.to_numeric(series, errors="coerce")
    return s.dropna().astype("int64")

def load_mrna_tss_flexible(path):
    """
    Accepts:
      - explicit TSS column: 'tss', 'ann_tss', 'pos', 'position', 'site'
      - or start/end (+ strand) to compute TSS
    Returns dict: chr -> np.int64 array
    """
    df = read_any(path)
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Identify key columns
    chr_col = next((c for c in df.columns if c in ("chr","chrom","chromosome","seqid")), None)
    strand_col = next((c for c in df.columns if c in ("strand","str")), None)
    tss_col = next((c for c in df.columns if c in ("tss","ann_tss","pos","position","site","tx_start","start_site")), None)
    start_col = "start" if "start" in df.columns else None
    end_col   = "end"   if "end"   in df.columns else None

    if chr_col is None:
        raise ValueError(f"Could not find a chromosome column in {path}")

    if tss_col is not None:
        tss = coerce_int(df[tss_col])
        keep = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # Fall back to start/end (+ strand)
    if start_col is not None and end_col is not None and strand_col is not None:
        start = coerce_int(df[start_col])
        end   = coerce_int(df[end_col])
        # align by index intersection
        common = start.index.intersection(end.index)
        start = start.loc[common]
        end   = end.loc[common]
        strand = df[strand_col].astype(str).str.strip().loc[common]
        tss = np.where(strand.values == "+", start.values, end.values).astype("int64")
        chrs = df[chr_col].astype(str).loc[common].values
        out = {}
        for chrom in np.unique(chrs):
            out[str(chrom)] = tss[chrs == chrom]
        return out

    # If only start present, use it as TSS
    if start_col is not None:
        tss = coerce_int(df[start_col])
        keep = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # If only a generic position column present
    if "pos" in df.columns:
        tss = coerce_int(df["pos"])
        keep = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    raise ValueError(f"Could not identify TSS columns in {path}. Columns: {list(df.columns)}")

def load_neg_tss_flexible(path):
    """
    Negatives with unknown/variable columns.
      - 5+ cols: chr col2 col3 name strand  -> TSS = col2 if '+' else col3
      - 4 cols : chr pos name strand        -> TSS = pos
      - 3 cols : chr pos strand             -> TSS = pos
      - 2 cols : chr pos                    -> TSS = pos
    Returns dict: chr -> np.int64 array.
    """
    df = read_any(path)
    ncol = df.shape[1]
    df.columns = [str(c).strip().lower() for c in df.columns]

    def group_out(chr_series, tss_series):
        tss = coerce_int(tss_series)
        keep_chr = chr_series.iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # Heuristics by column count / names
    if ncol >= 5 and {"chr","col2","col3","strand"}.issubset(df.columns):
        return group_out(df["chr"], np.where(df["strand"].astype(str) == "+", df["col2"], df["col3"]))
    if ncol >= 4 and {"chr","pos","strand"}.issubset(df.columns):
        return group_out(df["chr"], df["pos"])
    if {"chr","pos"}.issubset(df.columns):
        return group_out(df["chr"], df["pos"])

    # Try positional fallback
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str, comment="#")
    ncol = df.shape[1]
    if ncol >= 5:
        df = df.iloc[:, :5]; df.columns = ["chr","col2","col3","name","strand"]
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol == 4:
        df.columns = ["chr","pos","name","strand"]; return group_out(df["chr"], df["pos"])
    if ncol == 3:
        df.columns = ["chr","pos","strand"];        return group_out(df["chr"], df["pos"])
    if ncol == 2:
        df.columns = ["chr","pos"];                 return group_out(df["chr"], df["pos"])

    raise ValueError(f"Unexpected column layout in negatives: {path}")

# ================= Load coordinates =================
mRNA_tss_by_chr = load_mrna_tss_flexible(mRNA_file)
neg_tss_by_chr  = load_neg_tss_flexible(neg_file)

chroms_needed = set(mRNA_tss_by_chr) | set(neg_tss_by_chr)

# ================= Load CAGE once (fast) =================
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr == 0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
# collapse duplicates and build lookup per chr
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}

# ================= Accumulate mean coverage around TSS (±1000) =================
sum_mrna = np.zeros(OFFSETS.size, dtype=np.float64)
sum_neg  = np.zeros(OFFSETS.size, dtype=np.float64)
cnt_mrna = np.zeros(OFFSETS.size, dtype=np.int64)
cnt_neg  = np.zeros(OFFSETS.size, dtype=np.int64)

def accumulate(tss_array, series, sum_vec, cnt_vec):
    if series is None:
        return
    tss_array = np.asarray(tss_array, dtype=np.int64)
    if tss_array.size == 0:
        return
    pos_mat = tss_array[:, None] + OFFSETS[None, :]  # shape: n_sites x (2*WIN+1)
    vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
    sum_vec += np.nansum(vals, axis=0)
    cnt_vec += np.sum(~np.isnan(vals), axis=0)

for chr_name in chroms_needed:
    s = cage_by_chr.get(chr_name)
    accumulate(mRNA_tss_by_chr.get(chr_name, np.array([], dtype=np.int64)), s, sum_mrna, cnt_mrna)
    accumulate(neg_tss_by_chr.get(chr_name,  np.array([], dtype=np.int64)), s, sum_neg,  cnt_neg)

mean_mrna = np.divide(sum_mrna, cnt_mrna, out=np.zeros_like(sum_mrna), where=cnt_mrna > 0)
mean_neg  = np.divide(sum_neg,  cnt_neg,  out=np.zeros_like(sum_neg),  where=cnt_neg > 0)

# ================= Density helpers =================
def to_density(y):
    """Convert a nonnegative vector to a density (area = 1)."""
    y = np.asarray(y, dtype=float)
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    """
    Simple Gaussian smoothing with no external deps.
    sd_bp: standard deviation in bp.
    step_bp: sampling step in bp (here it's 1 because OFFSETS increments by 1).
    """
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    # Build kernel ~ +/- 4 SD
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

# ================= Convert to density (and smooth) =================
DENSITY_SMOOTH_SD = 25  # bp; set to 0 to disable smoothing

dens_mrna = to_density(mean_mrna)
dens_neg  = to_density(mean_neg)

if DENSITY_SMOOTH_SD and DENSITY_SMOOTH_SD > 0:
    dens_mrna = gaussian_smooth(dens_mrna, sd_bp=DENSITY_SMOOTH_SD, step_bp=1.0)
    dens_neg  = gaussian_smooth(dens_neg,  sd_bp=DENSITY_SMOOTH_SD, step_bp=1.0)

# ================= Save profiles (include densities) =================
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_CAGE_mRNA": mean_mrna,
    "mean_CAGE_Neg":  mean_neg,
    "density_mRNA":   dens_mrna,
    "density_Neg":    dens_neg,
    "n_mRNA":         cnt_mrna,
    "n_Neg":          cnt_neg
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)


# ================= Plot: density (new) =================
plt.figure(figsize=(7.5, 5))
plt.plot(OFFSETS, dens_mrna, label="mRNA (predicted)")
plt.plot(OFFSETS, dens_neg,  label="intergenic")
plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Density")
plt.legend(frameon=False)
plt.title("CAGE-seq coverage ±500 bp around TSS")
plt.tight_layout()
plt.savefig(plot_path_d, dpi=300)
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# Inputs (edit as needed)
# ==========================
refined_file   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
predicted_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/protein_coding_TSS.tsv"
telomere_neg   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"

gff_v48 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"
gff_v68 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmnDB_v68/PlasmoDB-68_Pfalciparum3D7.gff"

CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir       = os.path.dirname(CAGE_file)
plot_path     = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_density_0to1.png")
profiles_tsv  = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_profiles.tsv")

# Window around TSS
WIN = 1000
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)  # [-1000..1000], length 2001


# ==========================
# Utils
# ==========================
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def coerce_int(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.dropna().astype("int64")

def to_prob_density(y):
    """Probability density: nonnegative and sums to 1 over the window."""
    y = np.asarray(y, dtype=float).copy()
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def to_01(y):
    """Rescale to 0..1 by dividing by the max (for visualization with y-lim 0..1)."""
    y = np.asarray(y, dtype=float)
    m = y.max() if y.size else 0.0
    return y / m if m > 0 else y


# ==========================
# Loaders
# ==========================
def load_refined_tss_like_mrna(path):
    """
    Extract TSS from refined_file using the same 'mRNA-file method':
      TSS = start if strand '+', else end.
    Accepts CSV/TSV with columns: chr, start, end, gene, strand (or headerless in that order).
    Returns dict: chr -> np.int64 array of TSS.
    """
    try:
        # Try with header (comma or tab/space); if header doesn't match, fall back below
        df = pd.read_csv(path, dtype=str)
        cols = {c.lower(): c for c in df.columns}
        chr_col    = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
        start_col  = cols.get("start") or cols.get("tx_start")
        end_col    = cols.get("end")   or cols.get("tx_end")
        strand_col = cols.get("strand") or cols.get("str")
        if not (chr_col and start_col and end_col and strand_col):
            raise ValueError("Header present but required columns not found; using fallback.")
    except Exception:
        # Fallback: no/unknown header → assume 5 columns: chr start end gene strand
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", header=None, dtype=str)
        if df.shape[1] < 5:
            raise ValueError("refined_file needs at least 5 columns: chr start end gene strand.")
        df.columns = ["chr","start","end","gene","strand"]
        chr_col, start_col, end_col, strand_col = "chr","start","end","strand"
    else:
        # Normalize to standard colnames for downstream
        df = df.rename(columns={
            chr_col: "chr",
            start_col: "start",
            end_col: "end",
            strand_col: "strand"
        })

    # Compute TSS by strand rule
    df["start"] = coerce_int(df["start"])
    df["end"]   = coerce_int(df["end"])
    common = df["start"].dropna().index.intersection(df["end"].dropna().index)
    df = df.loc[common]
    df["tss"] = np.where(df["strand"].astype(str).str.strip() == "+", df["start"], df["end"])

    tss_by_chr = {c: g["tss"].astype("int64").to_numpy() for c, g in df.groupby("chr")}
    return tss_by_chr


def load_predicted_tss(path):
    """
    Predicted protein_coding_TSS.tsv (flexible).
    Tries columns in order: tss_pos, ann_tss, pos/position/site; otherwise start/end+strand.
    """
    df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    cols = {c.lower(): c for c in df.columns}
    chr_col = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
    if chr_col is None:
        raise ValueError("predicted_file: missing chr/seqid column.")

    tss_col = cols.get("tss_pos") or cols.get("ann_tss") or cols.get("pos") or cols.get("position") or cols.get("site")
    if tss_col:
        tss = coerce_int(df[tss_col])
        keep_chr = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    # fallback: start/end + strand
    strand_col = cols.get("strand") or cols.get("str")
    start_col  = cols.get("start")  or cols.get("tx_start")
    end_col    = cols.get("end")    or cols.get("tx_end")
    if not (strand_col and start_col and end_col):
        raise ValueError("predicted_file: could not identify TSS columns.")
    start = coerce_int(df[start_col]); end = coerce_int(df[end_col])
    common = start.index.intersection(end.index)
    strand = df[strand_col].astype(str).str.strip().loc[common]
    tss = np.where(strand.values == "+", start.loc[common].values, end.loc[common].values).astype("int64")
    chrs = df[chr_col].astype(str).loc[common].values
    out = {}
    for chrom in np.unique(chrs):
        out[str(chrom)] = tss[chrs == chrom]
    return out


def parse_gff_tss(path, prefer_protein_coding=True):
    """
    Parse GFF (PlasmoDB v48/v68). Use 'gene' features by default.
    TSS = start for '+' and end for '-'.
    If prefer_protein_coding=True, filter attributes containing 'protein' (case-insensitive).
    """
    col_names = ["seqid","source","type","start","end","score","strand","phase","attributes"]
    df = pd.read_csv(path, sep="\t", comment="#", header=None, names=col_names,
                     dtype={"seqid":str, "strand":str}, low_memory=False)

    df = df[df["type"].isin(["gene","coding_gene","mRNA","transcript"])]
    if df.empty:
        raise ValueError(f"GFF has no gene-like rows: {path}")

    if prefer_protein_coding and "attributes" in df.columns:
        attrs = df["attributes"].fillna("").astype(str).str.lower()
        mask = (
            attrs.str.contains("protein_coding") |
            attrs.str.contains("biotype=protein") |
            attrs.str.contains("gene_biotype=protein") |
            attrs.str.contains("gene_type=protein") |
            attrs.str.contains("transcript_type=protein") |
            attrs.str.contains(" protein ")
        )
        if mask.any():
            df = df[mask]

    start_i = coerce_int(df["start"])
    end_i   = coerce_int(df["end"])
    common  = start_i.index.intersection(end_i.index)
    df_sub  = df.loc[common]
    strand  = df_sub["strand"].astype(str).str.strip()
    tss = np.where(strand.values == "+", start_i.loc[common].values, end_i.loc[common].values).astype("int64")
    chrs = df_sub["seqid"].astype(str).values

    out = {}
    for chrom in np.unique(chrs):
        out[str(chrom)] = tss[chrs == chrom]
    return out


def load_telomere_negatives(path):
    """
    Flexible loader for random TSS from telomeres.
    Accepts:
      - 5+ cols: chr col2 col3 name strand  -> TSS = col2 if '+' else col3
      - 4 cols : chr pos name strand        -> TSS = pos
      - 3 cols : chr pos strand             -> TSS = pos
      - 2 cols : chr pos                    -> TSS = pos
    Header is optional.
    """
    try:
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
        df.columns = [str(c).strip().lower() for c in df.columns]
    except Exception:
        df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)

    def group_out(chr_series, tss_series):
        tss = coerce_int(tss_series)
        keep_chr = chr_series.iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out

    if isinstance(df.columns[0], str):  # named columns path
        ncol = df.shape[1]
        if ncol >= 5 and {"chr","col2","col3","strand"}.issubset(df.columns):
            return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
        if ncol >= 4 and {"chr","pos","strand"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
        if {"chr","pos"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
        # fall through to positional

    # Positional fallback
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    ncol = df.shape[1]
    if ncol >= 5:
        df = df.iloc[:, :5]; df.columns = ["chr","col2","col3","name","strand"]
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol == 4:
        df.columns = ["chr","pos","name","strand"]; return group_out(df["chr"], df["pos"])
    if ncol == 3:
        df.columns = ["chr","pos","strand"];        return group_out(df["chr"], df["pos"])
    if ncol == 2:
        df.columns = ["chr","pos"];                 return group_out(df["chr"], df["pos"])
    raise ValueError(f"Unexpected column layout in negatives: {path}")


# ==========================
# Build TSS sets (5 groups)
# ==========================
tss_refined   = load_refined_tss_like_mrna(refined_file)  # <-- uses the mRNA method (start/+ vs end/-)
tss_predicted = load_predicted_tss(predicted_file)
tss_v48       = parse_gff_tss(gff_v48, prefer_protein_coding=True)
tss_v68       = parse_gff_tss(gff_v68, prefer_protein_coding=True)
tss_telomere  = load_telomere_negatives(telomere_neg)

groups = {
    "Refined":       tss_refined,
    "Predicted":     tss_predicted,
    "PlasmoDB_v48":  tss_v48,
    "PlasmoDB_v68":  tss_v68,
    "Telomeres":     tss_telomere,
}


# ==========================
# Load CAGE (once)
# ==========================
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr == 0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
# Collapse duplicates and build lookup
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}


# ==========================
# Accumulate mean coverage
# ==========================
def accumulate_group(tss_by_chr, cage_by_chr):
    sum_vec = np.zeros(OFFSETS.size, dtype=np.float64)
    cnt_vec = np.zeros(OFFSETS.size, dtype=np.int64)
    for chr_name, tss_array in tss_by_chr.items():
        s = cage_by_chr.get(chr_name)
        if s is None:
            continue
        tss_array = np.asarray(tss_array, dtype=np.int64)
        if tss_array.size == 0:
            continue
        pos_mat = tss_array[:, None] + OFFSETS[None, :]
        vals = s.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
        sum_vec += np.nansum(vals, axis=0)
        cnt_vec += np.sum(~np.isnan(vals), axis=0)
    mean_vec = np.divide(sum_vec, cnt_vec, out=np.zeros_like(sum_vec), where=cnt_vec > 0)
    return mean_vec, cnt_vec

means = {}
counts = {}
for name, tss_by_chr in groups.items():
    m, n = accumulate_group(tss_by_chr, cage_by_chr)
    means[name]  = m
    counts[name] = n


# ==========================
# Build profiles table
# ==========================
df = pd.DataFrame({"offset_bp": OFFSETS})
for name, m in means.items():
    dens_prob = to_prob_density(m)     # sums to 1 over the window
    dens_01   = to_01(dens_prob)       # 0..1 for plotting
    df[f"mean_{name}"]      = m
    df[f"density_{name}"]   = dens_prob
    df[f"density01_{name}"] = dens_01

df.to_csv(profiles_tsv, sep="\t", index=False)


# ==========================
# Plot (five curves, 0..1)
# ==========================
plt.figure(figsize=(8.5, 5.5))
for name in groups.keys():
    plt.plot(OFFSETS, df[f"density01_{name}"], label=name)
plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Density (0–1 scaled)")
plt.title("CAGE density (±1000 bp) around TSS — Refined (mRNA method), Predicted, v48, v68, Telomeres")
plt.ylim(0, 1)
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# Inputs (edit as needed)
# ==========================
refined_file   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
predicted_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/protein_coding_TSS.tsv"
telomere_neg   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_intergenic.tsv"

# NOTE: double-check the v68 path spelling ('PlasmoDB', not 'PlasmnDB')
gff_v48 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"
gff_v68 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmnDB_v68/PlasmoDB-68_Pfalciparum3D7.gff"

CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir       = os.path.dirname(CAGE_file)
plot_path     = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_density_0to1.png")
profiles_tsv  = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_profiles.tsv")

# ==========================
# Parameters
# ==========================
WIN = 500                           # window half-size
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)  # [-1000..1000] length 2001
SMOOTH_SD_BP = 25                    # Gaussian smoothing SD in bp (set 0 to disable)
PLOT_BIN = 1                         # optional binning for display (e.g., 10)
CHUNK_SIZE = 20000                   # TSS chunk size to bound memory

# ==========================
# Utils
# ==========================
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def coerce_int(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.dropna().astype("int64")

def to_prob_density(y):
    """Probability density: nonnegative and sums to 1 over the window."""
    y = np.asarray(y, dtype=float).copy()
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def to_01(y):
    """Rescale to 0..1 by dividing by the max (for visualization)."""
    y = np.asarray(y, dtype=float)
    m = y.max() if y.size else 0.0
    return y / m if m > 0 else y

def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

# ==========================
# Loaders
# ==========================
def load_refined_tss_like_mrna(path):
    """
    Extract TSS using the mRNA-file rule: TSS = start on '+' and end on '-'.
    Accepts headered CSV/TSV with chr/start/end/gene/strand; or headerless in that order.
    Returns: dict chr -> np.int64 array
    """
    try:
        df = pd.read_csv(path, dtype=str)
        cols = {c.lower(): c for c in df.columns}
        chr_col    = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
        start_col  = cols.get("start") or cols.get("tx_start")
        end_col    = cols.get("end")   or cols.get("tx_end")
        strand_col = cols.get("strand") or cols.get("str")
        if not (chr_col and start_col and end_col and strand_col):
            raise ValueError("Missing required columns; fallback to headerless parse.")
        df = df.rename(columns={chr_col:"chr", start_col:"start", end_col:"end", strand_col:"strand"})
    except Exception:
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", header=None, dtype=str)
        if df.shape[1] < 5:
            raise ValueError("refined_file needs ≥5 cols: chr start end gene strand.")
        df.columns = ["chr","start","end","gene","strand"]

    df["start"] = coerce_int(df["start"])
    df["end"]   = coerce_int(df["end"])
    idx = df["start"].dropna().index.intersection(df["end"].dropna().index)
    df = df.loc[idx]
    df["tss"] = np.where(df["strand"].astype(str).str.strip() == "+", df["start"], df["end"])
    return {c: g["tss"].astype("int64").to_numpy() for c, g in df.groupby("chr")}

def load_predicted_tss(path):
    df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    cols = {c.lower(): c for c in df.columns}
    chr_col = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
    if chr_col is None:
        raise ValueError("predicted_file: missing chr/seqid column.")
    tss_col = cols.get("tss_pos") or cols.get("ann_tss") or cols.get("pos") or cols.get("position") or cols.get("site")
    if tss_col:
        tss = coerce_int(df[tss_col])
        keep_chr = df[chr_col].iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out
    strand_col = cols.get("strand") or cols.get("str")
    start_col  = cols.get("start")  or cols.get("tx_start")
    end_col    = cols.get("end")    or cols.get("tx_end")
    if not (strand_col and start_col and end_col):
        raise ValueError("predicted_file: could not identify TSS columns.")
    start = coerce_int(df[start_col]); end = coerce_int(df[end_col])
    common = start.index.intersection(end.index)
    strand = df[strand_col].astype(str).str.strip().loc[common]
    tss = np.where(strand.values == "+", start.loc[common].values, end.loc[common].values).astype("int64")
    chrs = df[chr_col].astype(str).loc[common].values
    out = {}
    for chrom in np.unique(chrs):
        out[str(chrom)] = tss[chrs == chrom]
    return out

def parse_gff_tss(path, prefer_protein_coding=True):
    col_names = ["seqid","source","type","start","end","score","strand","phase","attributes"]
    df = pd.read_csv(path, sep="\t", comment="#", header=None, names=col_names,
                     dtype={"seqid":str, "strand":str}, low_memory=False)
    df = df[df["type"].isin(["gene","coding_gene","mRNA","transcript"])]
    if df.empty:
        raise ValueError(f"GFF has no gene-like rows: {path}")
    if prefer_protein_coding and "attributes" in df.columns:
        attrs = df["attributes"].fillna("").astype(str).str.lower()
        mask = (
            attrs.str.contains("protein_coding") |
            attrs.str.contains("biotype=protein") |
            attrs.str.contains("gene_biotype=protein") |
            attrs.str.contains("gene_type=protein") |
            attrs.str.contains("transcript_type=protein") |
            attrs.str.contains(" protein ")
        )
        if mask.any():
            df = df[mask]
    start_i = coerce_int(df["start"])
    end_i   = coerce_int(df["end"])
    common  = start_i.index.intersection(end_i.index)
    df_sub  = df.loc[common]
    strand  = df_sub["strand"].astype(str).str.strip()
    tss = np.where(strand.values == "+", start_i.loc[common].values, end_i.loc[common].values).astype("int64")
    chrs = df_sub["seqid"].astype(str).values
    out = {}
    for chrom in np.unique(chrs):
        out[str(chrom)] = tss[chrs == chrom]
    return out

def load_telomere_negatives(path):
    try:
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
        df.columns = [str(c).strip().lower() for c in df.columns]
    except Exception:
        df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    def group_out(chr_series, tss_series):
        tss = coerce_int(tss_series)
        keep_chr = chr_series.iloc[tss.index].astype(str)
        out = {}
        for chrom, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(chrom)] = sub["tss"].to_numpy(dtype=np.int64)
        return out
    if isinstance(df.columns[0], str):
        ncol = df.shape[1]
        if ncol >= 5 and {"chr","col2","col3","strand"}.issubset(df.columns):
            return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
        if ncol >= 4 and {"chr","pos","strand"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
        if {"chr","pos"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    ncol = df.shape[1]
    if ncol >= 5:
        df = df.iloc[:, :5]; df.columns = ["chr","col2","col3","name","strand"]
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol == 4:
        df.columns = ["chr","pos","name","strand"]; return group_out(df["chr"], df["pos"])
    if ncol == 3:
        df.columns = ["chr","pos","strand"];        return group_out(df["chr"], df["pos"])
    if ncol == 2:
        df.columns = ["chr","pos"];                 return group_out(df["chr"], df["pos"])
    raise ValueError(f"Unexpected column layout in negatives: {path}")

# ==========================
# Build TSS sets (5 groups)
# ==========================
tss_refined   = load_refined_tss_like_mrna(refined_file)
tss_predicted = load_predicted_tss(predicted_file)
tss_v48       = parse_gff_tss(gff_v48, prefer_protein_coding=True)
tss_v68       = parse_gff_tss(gff_v68, prefer_protein_coding=True)
tss_telomere  = load_telomere_negatives(telomere_neg)

groups = {
    "Refined":       tss_refined,
    "Predicted":     tss_predicted,
    "PlasmoDB_v48":  tss_v48,
    "PlasmoDB_v68":  tss_v68,
    "Telomeres":     tss_telomere,
}

# ==========================
# Load CAGE once
# ==========================
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr == 0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}

# ==========================
# Accumulate mean coverage (fill missing as 0; chunked)
# ==========================
def accumulate_group(tss_by_chr, cage_by_chr, offsets, chunk_size=20000):
    sum_vec = np.zeros(offsets.size, dtype=np.float64)
    n_sites_total = 0

    for chr_name, tss_array in tss_by_chr.items():
        s = cage_by_chr.get(chr_name)
        if s is None:
            continue
        tss_array = np.asarray(tss_array, dtype=np.int64)
        if tss_array.size == 0:
            continue
        n_sites_total += tss_array.size

        for i in range(0, tss_array.size, chunk_size):
            chunk = tss_array[i:i+chunk_size]
            pos_mat = chunk[:, None] + offsets[None, :]
            vals = s.reindex(pos_mat.ravel(), fill_value=0.0).to_numpy().reshape(pos_mat.shape)
            sum_vec += vals.sum(axis=0)

    # every site contributes at every offset (we filled missing with 0)
    cnt_vec = np.full(offsets.size, n_sites_total, dtype=np.int64)
    mean_vec = np.divide(sum_vec, cnt_vec, out=np.zeros_like(sum_vec), where=cnt_vec > 0)
    return mean_vec, cnt_vec

means = {}
counts = {}
for name, tss_by_chr in groups.items():
    m, n = accumulate_group(tss_by_chr, cage_by_chr, OFFSETS, CHUNK_SIZE)
    means[name]  = m
    counts[name] = n

# ==========================
# Build profiles table (smooth then renormalize; optional binning)
# ==========================
df = pd.DataFrame({"offset_bp": OFFSETS})
for name, m in means.items():
    d = to_prob_density(m)
    if SMOOTH_SD_BP > 0:
        d = gaussian_smooth(d, sd_bp=SMOOTH_SD_BP, step_bp=1.0)
        d = to_prob_density(d)      # renormalize after smoothing
    df[f"mean_{name}"]      = m
    df[f"density_{name}"]   = d
    df[f"density01_{name}"] = to_01(d)

# ---------- Weighted density (shape × magnitude) ----------
# Mass = total mean CPM across the ±1000 bp window for each group
masses = {name: float(np.nansum(means[name])) for name in groups}
max_mass = max(masses.values()) if masses else 0.0

# Build weighted densities (do NOT rescale per-curve; use a single global scale for plotting)
df_w = pd.DataFrame({"offset_bp": df["offset_bp"]})
for name in groups.keys():
    weight = (masses[name] / max_mass) if max_mass > 0 else 0.0  # 0..1 per group
    wd = df[f"density_{name}"].to_numpy() * weight               # shape × magnitude
    df_w[f"weighted_density_{name}"] = wd

# Save weighted densities to TSV (raw, not 0–1 display-scaled)
wd_tsv = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_weighted_density.tsv")
df_w.to_csv(wd_tsv, sep="\t", index=False)

# Global 0–1 scaling ACROSS groups for display only
global_max = np.max([df_w[f"weighted_density_{name}"].max() for name in groups.keys()]) if groups else 0.0

plt.figure(figsize=(8.5, 5.5))
for name in groups.keys():
    y = df_w[f"weighted_density_{name}"].to_numpy()
    y_plot = y / global_max if global_max > 0 else y
    plt.plot(df_w["offset_bp"], y_plot, label=name)

plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Weighted density (0–1 across groups)")
plt.title("CAGE weighted density (±500 bp) around TSS — shape × magnitude")
plt.ylim(0, 1)
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "CAGE_TSS_pm1000_5way_weighted_density.png"), dpi=150)
plt.show()

# (Optional) Inspect relative masses
for k, v in masses.items():
    print(f"{k:14s} total_mass(sum(mean)) = {v:.6g}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# Inputs (edit as needed)
# ==========================
refined_file   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
# CHANGED: use the BED file for predicted TSS
predicted_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/predicted_gene_boundaries.bed"
telomere_neg   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"

gff_v48 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"
gff_v68 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmnDB_v68/PlasmoDB-68_Pfalciparum3D7.gff"

CAGE_file = "/rhome/zli529/lab/SRA_toolkit/cage_seq_data/paired-end/aligned_hisat2_pe/readcounts/summed_stage_means_cpm.txt"

out_dir = os.path.dirname(CAGE_file)
plot_weighted_path = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_weighted_density.png")
weighted_tsv       = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_weighted_density.tsv")
profiles_tsv       = os.path.join(out_dir, "CAGE_TSS_pm1000_5way_profiles.tsv")

# ==========================
# Parameters
# ==========================
WIN = 1000
OFFSETS = np.arange(-WIN, WIN + 1, dtype=np.int64)    # [-1000..1000]
SMOOTH_SD_BP = 25                                     # set 0 to disable
PLOT_BIN = 1                                          # e.g., 10 for extra smoothing
CHUNK_SIZE = 20000

# ==========================
# Utils
# ==========================
def cage_has_header(path):
    with open(path) as f:
        first = f.readline().strip().split()
    return bool(first) and first[0].lower().startswith("chr")

def coerce_int(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.dropna().astype("int64")

def to_prob_density(y):
    y = np.asarray(y, dtype=float).copy()
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def to_01(y):
    y = np.asarray(y, dtype=float)
    m = y.max() if y.size else 0.0
    return y / m if m > 0 else y

def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

# ==========================
# Loaders
# ==========================
def load_refined_tss_like_mrna(path):
    """
    mRNA-style rule: TSS = start if '+' else end.
    Accepts headered CSV/TSV (chr/start/end/gene/strand) or headerless with that order.
    """
    try:
        df = pd.read_csv(path, dtype=str)
        cols = {c.lower(): c for c in df.columns}
        chr_col    = cols.get("chr") or cols.get("chrom") or cols.get("chromosome") or cols.get("seqid")
        start_col  = cols.get("start") or cols.get("tx_start")
        end_col    = cols.get("end")   or cols.get("tx_end")
        strand_col = cols.get("strand") or cols.get("str")
        if not (chr_col and start_col and end_col and strand_col):
            raise ValueError("Missing columns; try headerless parse.")
        df = df.rename(columns={chr_col:"chr", start_col:"start", end_col:"end", strand_col:"strand"})
    except Exception:
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", header=None, dtype=str)
        if df.shape[1] < 5:
            raise ValueError("refined_file needs ≥5 columns: chr start end gene strand.")
        df.columns = ["chr","start","end","gene","strand"]
    df["start"] = coerce_int(df["start"])
    df["end"]   = coerce_int(df["end"])
    idx = df["start"].dropna().index.intersection(df["end"].dropna().index)
    df = df.loc[idx]
    df["tss"] = np.where(df["strand"].astype(str).str.strip() == "+", df["start"], df["end"])
    return {c: g["tss"].astype("int64").to_numpy() for c, g in df.groupby("chr")}

def load_predicted_tss_from_bed(path):
    """
    BED parser (0-based, half-open). We convert to 1-based coordinates to match CAGE/GFF.
    TSS (1-based) = chromStart+1 on '+' strand; chromEnd on '-' strand.
    Accepts 3+ column BED; if strand is missing, assumes '+'.
    """
    # Read BED (skip comments; handle 'track'/'browser' lines after read)
    df = pd.read_csv(path, sep=r"\s+|\t|,", engine="python", header=None, comment="#", dtype=str)
    if df.empty:
        return {}
    # Drop 'track'/'browser' rows if present
    bad = df[0].astype(str).str.lower().isin(["track", "browser"])
    if bad.any():
        df = df[~bad]

    if df.shape[1] < 3:
        raise ValueError("BED must have at least 3 columns: chrom, start, end.")

    chrom  = df.iloc[:, 0].astype(str)
    start0 = coerce_int(df.iloc[:, 1])  # 0-based
    end0   = coerce_int(df.iloc[:, 2])  # 0-based exclusive
    common = start0.index.intersection(end0.index)

    if df.shape[1] >= 6:
        strand = df.iloc[:, 5].astype(str).str.strip().loc[common]
        # 1-based TSS: + => start+1 ; - => end
        tss1 = np.where(strand.values == "+", start0.loc[common].values + 1, end0.loc[common].values).astype("int64")
    else:
        # no strand provided → assume '+'
        strand = pd.Series("+", index=common)
        tss1 = (start0.loc[common].values + 1).astype("int64")

    chrs = chrom.loc[common].values
    out = {}
    for c in np.unique(chrs):
        out[str(c)] = tss1[chrs == c]
    return out

def parse_gff_tss(path, prefer_protein_coding=True):
    col_names = ["seqid","source","type","start","end","score","strand","phase","attributes"]
    df = pd.read_csv(path, sep="\t", comment="#", header=None, names=col_names,
                     dtype={"seqid":str, "strand":str}, low_memory=False)
    df = df[df["type"].isin(["gene","coding_gene","mRNA","transcript"])]
    if df.empty:
        return {}
    if prefer_protein_coding and "attributes" in df.columns:
        attrs = df["attributes"].fillna("").astype(str).str.lower()
        mask = (
            attrs.str.contains("protein_coding") |
            attrs.str.contains("biotype=protein") |
            attrs.str.contains("gene_biotype=protein") |
            attrs.str.contains("gene_type=protein") |
            attrs.str.contains("transcript_type=protein") |
            attrs.str.contains(" protein ")
        )
        if mask.any():
            df = df[mask]
    start_i = coerce_int(df["start"])
    end_i   = coerce_int(df["end"])
    common  = start_i.index.intersection(end_i.index)
    df_sub  = df.loc[common]
    strand  = df_sub["strand"].astype(str).str.strip()
    tss = np.where(strand.values == "+", start_i.loc[common].values, end_i.loc[common].values).astype("int64")
    chrs = df_sub["seqid"].astype(str).values
    out = {}
    for c in np.unique(chrs):
        out[str(c)] = tss[chrs == c]
    return out

def load_telomere_negatives(path):
    try:
        df = pd.read_csv(path, sep=r"\s+|,|\t", engine="python", dtype=str)
        df.columns = [str(c).strip().lower() for c in df.columns]
    except Exception:
        df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    def group_out(chr_series, tss_series):
        tss = coerce_int(tss_series)
        keep_chr = chr_series.iloc[tss.index].astype(str)
        out = {}
        for c, sub in pd.DataFrame({"chr": keep_chr, "tss": tss}).groupby("chr"):
            out[str(c)] = sub["tss"].to_numpy(dtype=np.int64)
        return out
    if isinstance(df.columns[0], str):
        ncol = df.shape[1]
        if ncol >= 5 and {"chr","col2","col3","strand"}.issubset(df.columns):
            return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
        if ncol >= 4 and {"chr","pos","strand"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
        if {"chr","pos"}.issubset(df.columns):
            return group_out(df["chr"], df["pos"])
    # positional fallback
    df = pd.read_csv(path, sep=r"\s+|,|\t", header=None, engine="python", dtype=str)
    ncol = df.shape[1]
    if ncol >= 5:
        df = df.iloc[:, :5]; df.columns = ["chr","col2","col3","name","strand"]
        return group_out(df["chr"], np.where(df["strand"].astype(str)=="+", df["col2"], df["col3"]))
    if ncol == 4:
        df.columns = ["chr","pos","name","strand"]; return group_out(df["chr"], df["pos"])
    if ncol == 3:
        df.columns = ["chr","pos","strand"];        return group_out(df["chr"], df["pos"])
    if ncol == 2:
        df.columns = ["chr","pos"];                 return group_out(df["chr"], df["pos"])
    raise ValueError(f"Unexpected column layout in negatives: {path}")

# ==========================
# Build TSS sets (5 groups)
# ==========================
tss_refined   = load_refined_tss_like_mrna(refined_file)
tss_predicted = load_predicted_tss_from_bed(predicted_file)   # <- BED-based predicted TSS
tss_v48       = parse_gff_tss(gff_v48, prefer_protein_coding=True)
tss_v68       = parse_gff_tss(gff_v68, prefer_protein_coding=True)
tss_telomere  = load_telomere_negatives(telomere_neg)

groups = {
    "Refined":       tss_refined,
    "Predicted":     tss_predicted,
    "PlasmoDB_v48":  tss_v48,
    "PlasmoDB_v68":  tss_v68,
    "Telomeres":     tss_telomere,
}

# ==========================
# Load CAGE once
# ==========================
hdr = 0 if cage_has_header(CAGE_file) else None
cage = pd.read_csv(
    CAGE_file, sep="\t", header=hdr, engine="c",
    names=None if hdr == 0 else ["chr","pos","sum_cpm"],
    usecols=[0,1,2],
    dtype={"chr":"string","pos":"int64","sum_cpm":"float64"}
)
cage = cage.groupby(["chr","pos"], as_index=False)["sum_cpm"].sum()
cage_by_chr = {c: pd.Series(g.sum_cpm.values, index=g.pos.values) for c,g in cage.groupby("chr")}

# ==========================
# Accumulate mean coverage (zeros for missing; chunked)
# ==========================
def accumulate_group(tss_by_chr, cage_by_chr, offsets, chunk_size=20000):
    sum_vec = np.zeros(offsets.size, dtype=np.float64)
    n_sites_total = 0
    for chr_name, tss_array in tss_by_chr.items():
        s = cage_by_chr.get(chr_name)
        if s is None:
            continue
        tss_array = np.asarray(tss_array, dtype=np.int64)
        if tss_array.size == 0:
            continue
        n_sites_total += tss_array.size
        for i in range(0, tss_array.size, chunk_size):
            chunk = tss_array[i:i+chunk_size]
            pos_mat = chunk[:, None] + offsets[None, :]
            vals = s.reindex(pos_mat.ravel(), fill_value=0.0).to_numpy().reshape(pos_mat.shape)
            sum_vec += vals.sum(axis=0)
    cnt_vec = np.full(offsets.size, n_sites_total, dtype=np.int64)
    mean_vec = np.divide(sum_vec, cnt_vec, out=np.zeros_like(sum_vec), where=cnt_vec > 0)
    return mean_vec, cnt_vec

means = {}
counts = {}
for name, tss_by_chr in groups.items():
    m, n = accumulate_group(tss_by_chr, cage_by_chr, OFFSETS, CHUNK_SIZE)
    means[name]  = m
    counts[name] = n

# ==========================
# Build profiles (densities)
# ==========================
df = pd.DataFrame({"offset_bp": OFFSETS})
for name, m in means.items():
    d = to_prob_density(m)
    if SMOOTH_SD_BP > 0:
        d = gaussian_smooth(d, sd_bp=SMOOTH_SD_BP, step_bp=1.0)
        d = to_prob_density(d)  # renormalize after smoothing
    df[f"mean_{name}"]      = m
    df[f"density_{name}"]   = d
    df[f"density01_{name}"] = to_01(d)

# Save means + densities
df.to_csv(profiles_tsv, sep="\t", index=False)

# ==========================
# Weighted density (shape × magnitude)
# ==========================
masses = {name: float(np.nansum(means[name])) for name in groups}
max_mass = max(masses.values()) if masses else 0.0

df_w = pd.DataFrame({"offset_bp": df["offset_bp"]})
for name in groups.keys():
    weight = (masses[name] / max_mass) if max_mass > 0 else 0.0
    df_w[f"weighted_density_{name}"] = df[f"density_{name}"].to_numpy() * weight

# Save raw weighted densities
df_w.to_csv(weighted_tsv, sep="\t", index=False)

# Global 0–1 scaling across groups for display
global_max = np.max([df_w[f"weighted_density_{name}"].max() for name in groups]) if groups else 0.0

plot_x = df_w["offset_bp"].to_numpy()
plot_series = {}
for name in groups.keys():
    y = df_w[f"weighted_density_{name}"].to_numpy()
    plot_series[name] = (y / global_max) if global_max > 0 else y

# Optional simple binning for display
if PLOT_BIN > 1:
    pad = (-len(plot_x)) % PLOT_BIN
    x_pad = np.pad(plot_x, (0, pad), mode="edge")
    plot_x = x_pad.reshape(-1, PLOT_BIN)[:, PLOT_BIN // 2]
    for name in list(plot_series.keys()):
        y_pad = np.pad(plot_series[name], (0, pad), mode="edge")
        plot_series[name] = y_pad.reshape(-1, PLOT_BIN).mean(axis=1)

# ==========================
# Plot (weighted density)
# ==========================
plt.figure(figsize=(8.5, 5.5))
for name in groups.keys():
    plt.plot(plot_x, plot_series[name], label=name)
plt.axvline(0, ls="--", lw=1)
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Weighted density (0–1 across groups)")
plt.title("CAGE weighted density (±1000 bp) around TSS — Refined, BED Predicted, v48, v68, Telomeres")
plt.ylim(0, 1)
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(plot_weighted_path, dpi=150)
plt.show()

# For quick sanity:
for k, v in masses.items():
    print(f"{k:14s} total_mass(sum(mean)) = {v:.6g}")
